In [5]:
import os
import sqlite3
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

# try:
#     BASE_DIR = os.path.dirname(__file__)
# except NameError:
#     BASE_DIR = os.getcwd()

# DB_PATH = os.path.join(BASE_DIR, "atliq_tshirts.db")

DB_PATH = os.path.join(os.getcwd(), "atliq_tshirts.db")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [11]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_schema  WHERE type='table';")
tables = cursor.fetchall()
print(tables)

# conn.close()

[('discounts',), ('sqlite_sequence',), ('t_shirts',)]


In [12]:
# --- Fetch Data for 't_shirts' Table ---
cursor.execute("PRAGMA table_info(t_shirts);")
tshirts_schema = cursor.fetchall()
cursor.execute("SELECT * FROM t_shirts;")
tshirts_data = cursor.fetchall()

# --- Fetch Data for 'discounts' Table ---
cursor.execute("PRAGMA table_info(discounts);")
discounts_schema = cursor.fetchall()
cursor.execute("SELECT * FROM discounts;")
discounts_data = cursor.fetchall()

conn.close() # Safely close database connection

In [13]:
db_context = f"""
TABLE NAME: t_shirts
Columns (id, name, type, notnull, dflt_value, pk): {tshirts_schema}
Data Rows: {tshirts_data}

TABLE NAME: discounts
Columns (id, name, type, notnull, dflt_value, pk): {discounts_schema}
Data Rows: {discounts_data}
"""

# 5. Build prompt incorporating both tables
question = "How many t-shirts do we have left for Nike in Extra Small size, and do they have any applicable discounts?"

prompt = f"""
You are a store inventory assistant. Analyze the database snapshot provided below to accurately answer the user's question by cross-referencing both tables if necessary.

Database Snapshot:
{db_context}

User Question: {question}
Answer clearly based ONLY on the data provided above.
"""

# 6. Execute and print response
response = llm.invoke(prompt)

print("\n--- Answer ---")
print(response.content)


--- Answer ---
To answer the user's question, we need to cross-reference the data from both tables.

From the t_shirts table, we can see that there is no Nike t-shirt in Extra Small (XS) size. The Nike t-shirts available are:

- Black, size L (t_shirt_id: 5)
- Blue, size XL (t_shirt_id: 6)

Since there are no Nike t-shirts in Extra Small size, we have 0 t-shirts left for Nike in Extra Small size.

As for applicable discounts, we can check the discounts table for t_shirt_id that matches the Nike t-shirts. The discounts for Nike t-shirts are:

- t_shirt_id: 5 (Black, size L) has a discount_id: 5 with a 25% discount
- t_shirt_id: 6 (Blue, size XL) has a discount_id: 6 with a 10% discount

However, since there are no Nike t-shirts in Extra Small size, there are no applicable discounts for Nike t-shirts in Extra Small size.

In summary, we have 0 Nike t-shirts in Extra Small size, and therefore, there are no applicable discounts for this specific size and brand.
